In [17]:
import napari
import tifffile
import numpy as np
from pathlib import Path
from magicgui import magicgui
from magicgui.widgets import TextEdit, PushButton, Container, ComboBox
from napari.qt.threading import thread_worker
import os
import tempfile
import numpy as np
from scipy.ndimage import distance_transform_edt

import matplotlib.pyplot as plt


In [18]:

def safe_imwrite(path, arr, *, imagej=True, resolution=None, resolutionunit=None,
                 metadata=None):
    """
    IMproved memory efficiency
    """
    path = os.fspath(path)
    arr = np.asarray(arr)

    # ImageJ TIFF only supports these dtypes -> fail loudly BEFORE touching disk.
    if imagej and arr.dtype not in (np.uint8, np.uint16, np.int16, np.float32):
        raise TypeError(
            f"imagej=True cannot store dtype {arr.dtype}. Cast first, e.g. "
            f"arr.astype(np.float32) (intensities) or arr.astype(np.uint16) (labels)."
        )

    folder = os.path.dirname(path) or "."
    fd, tmp = tempfile.mkstemp(suffix=".tmp.tif", dir=folder)
    os.close(fd)
    try:
        kwargs = {"imagej": imagej}
        if resolution is not None:
            kwargs["resolution"] = resolution
        if resolutionunit is not None:
            kwargs["resolutionunit"] = resolutionunit
        if metadata is not None:
            kwargs["metadata"] = metadata

        tifffile.imwrite(tmp, arr, **kwargs)

        # Verify the temp file matches the source before replacing it, using a
        # memory-map + per-slice comparison so we never allocate a whole extra
        # copy or a whole boolean array.
        written = tifffile.imread(tmp, out="memmap")
        try:
            ok = tuple(written.shape) == tuple(arr.shape)
            if ok and arr.ndim == 0:
                ok = bool(np.array_equal(written, arr))
            elif ok:
                for i in range(arr.shape[0]):
                    if not np.array_equal(written[i], arr[i]):
                        ok = False
                        break
        finally:
            # Close the memory-map's file handle before replace/remove, otherwise
            # os.replace/os.remove fails on Windows ("file in use").
            mm = getattr(written, "_mmap", None)
            if mm is not None:
                mm.close()
            del written
        if not ok:
            raise ValueError("verification failed: written data != source array")

        os.replace(tmp, path)  # atomic on the same filesystem
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)
    return path


In [19]:
def read_image_and_meta(path):
    """Read a tif as a (Z, C, Y, X) array along with the metadata needed to
    write it back out unchanged (pixel size / spacing / unit)."""
    with tifffile.TiffFile(path) as tif:
        series = max(tif.series, key=lambda s: s.size)
        arr = series.asarray()
        axes = series.axes
        ij_meta = dict(tif.imagej_metadata or {})

        def _rational(tag):
            if tag is None:
                return None
            val = tag.value
            if isinstance(val, tuple) and len(val) == 2:
                return val[0] / val[1] if val[1] else None
            return val

        xres = yres = resunit = None
        for page in tif.pages:
            if "XResolution" in page.tags:
                xres = _rational(page.tags.get("XResolution"))
                yres = _rational(page.tags.get("YResolution"))
                resunit_tag = page.tags.get("ResolutionUnit")
                resunit = int(resunit_tag.value) if resunit_tag is not None else None
                break

    meta = {
        "axes": axes,
        "imagej": ij_meta,
        "xres": xres,
        "yres": yres,
        "resunit": resunit,
    }
    return arr, meta



In [20]:

def save_image(path, arr, meta):
    """Write `arr` to `path`, preserving the original pixel size / spacing /
    unit metadata.

    The channel / slice / frame counts are recomputed from `arr` (rather than
    copied from the source metadata) so the ImageJ header stays correct even
    when the channel count changes, e.g. when we add a 4th `valid` channel to a
    previously 3-channel stack.

    Uses safe_imwrite so a failed write can never corrupt the existing file.
    """
    ij = meta["imagej"]
    axes = meta["axes"]
    md = {"axes": axes}

    for key in ("spacing", "unit", "finterval", "fps", "mode"):
        if key in ij:
            md[key] = ij[key]

    for ax, name in (("Z", "slices"), ("C", "channels"), ("T", "frames")):
        if ax in axes:
            md[name] = arr.shape[axes.index(ax)]

    kwargs = {"imagej": True, "metadata": md}
    if meta["xres"] and meta["yres"]:
        kwargs["resolution"] = (meta["xres"], meta["yres"])
    if meta["resunit"] is not None:
        kwargs["resolutionunit"] = meta["resunit"]

    safe_imwrite(path, arr, **kwargs)


In [21]:
class MaskCurator:
    """Napari GUI to curate the mask channel of a (Z, C, Y, X) tif.

    Saving strategy (optimised for very large stacks on slow/network drives):
      * While curating, ONLY the mask channel (single channel, ZYX) is autosaved
        to a small `<name>_MASK_AUTOSAVE.tif`, on a BACKGROUND THREAD (napari
        thread_worker) so the GUI stays responsive. The original is not touched.
      * "Save final" COMMITS: it folds the mask into the 3-channel stack, writes
        it OVER the original file, then deletes the now-obsolete `_CURATED` and
        `_MASK_AUTOSAVE` sidecars.
      * Closing the window writes the full 3-channel to a non-destructive
        `<name>_CURATED.tif` safety net -- but ONLY if there are unsaved mask
        edits; if nothing changed since the last save it skips the write.
      * On load it resumes from an existing `_CURATED`, and if a newer
        `_MASK_AUTOSAVE` exists it overlays that onto the mask channel.
    All writes go through safe_imwrite (atomic temp-then-replace), so a failed
    write can never corrupt an existing file.

    Memory notes (these images can be ~5600x5600): the mask is held as the
    smallest integer dtype that fits its labels (uint8/uint16) rather than int32,
    and the save paths avoid allocating a second full-size copy of the mask, so
    curating + saving these large stacks stays within RAM.

    Background saves use napari's thread_worker: the worker does the disk write
    off the GUI thread, and its returned/errored/finished signals fire back ON
    the GUI thread, so "save completed" / error messages appear in the Log panel.
    """

    AUTOSAVE_SECONDS = 300
    REGION_GRID_THRESHOLD = 1024
    MAX_HISTORY = 10  # max number of bulk operations that can be undone

    def __init__(self, default_folder=None,
                 brightfield_channel=0, mask_channel=1,
                 cell_fluo_channel=1, perfused_channel=None):
        self.default_folder = Path(default_folder) if default_folder else None
        # Channel roles. brightfield + mask ("existing label") are required; the
        # cell / perfused fluorescence channels are optional (None = not shown).
        self.brightfield_channel = brightfield_channel
        self.mask_channel = mask_channel
        self.cell_fluo_channel = cell_fluo_channel
        self.perfused_channel = perfused_channel
        self._num_channels = None      # channel count of the loaded stack
        self.channel_widget = None     # GUI to assign channel roles

        self.viewer = None
        self.base_path = None      # original identity used for naming outputs
        self.path = None           # file actually loaded (original or _CURATED)
        self.arr = None
        self.meta = None
        self.mask_layer = None
        self.log_widget = None
        self._orig_close = None
        self._autosave_timer = None
        self._save_worker = None   # currently running background save worker
        self._save_running = False
        self._dirty = False

        # Operation region controls (editing remains global; bulk ops can be scoped).
        self.region_widget = None
        self.operation_scope = "Full image"
        self.active_region = "R1"
        self._region_rows = 2
        self._region_cols = 2
        self._region_bounds = []
        self._region_outline_layer = None

        # Undo/redo history for the bulk operations (replace / interpolate /
        # fade out). Painting/erasing keep napari's own separate undo stack.
        self._undo_stack = []
        self._redo_stack = []
        self.undo_button = None
        self.redo_button = None
        self.history_widget = None

    def _log(self, message):
        """Append a message to the GUI log panel (falls back to print).

        Safe to call from the GUI thread. Background work is done in
        thread_workers whose signals are delivered on the GUI thread, so their
        callbacks may call this too.
        """
        if self.log_widget is not None:
            current = self.log_widget.value
            self.log_widget.value = (current + "\n" + message) if current else message
        else:
            print(message)

    @staticmethod
    def _rescale_to_255(image):
        """Linearly rescale an image so its min/max map to 0/255 (uint8)."""
        image = image.astype(np.float32)
        lo, hi = float(image.min()), float(image.max())
        if hi > lo:
            image = (image - lo) / (hi - lo) * 255.0
        else:
            image = np.zeros_like(image)
        return image.astype(np.uint8)

    @staticmethod
    def _labels_dtype(max_value):
        """Smallest ImageJ-friendly integer dtype that can hold `max_value`.

        Using uint8/uint16 instead of int32 for the mask layer cuts its RAM
        footprint by 4x / 2x respectively (e.g. 0.9 GB vs 3.6 GB for a
        31x5606x5605 stack), which is what prevented saving these large images.
        uint16 still allows up to 65535 distinct labels.
        """
        return np.uint8 if int(max_value) <= 255 else np.uint16

    @staticmethod
    def _contrast_limits(volume):
        """Min/max of a channel volume for a full-dynamic-range display stretch.

        Computed in a single pass (no full-size copy). Returns None when the
        volume is flat so napari falls back to its own default limits.
        """
        lo, hi = float(np.min(volume)), float(np.max(volume))
        return [lo, hi] if hi > lo else None

    def _layer_scale(self):
        """Return the (z, y, x) voxel scale derived from the loaded metadata.

        Z spacing comes from the ImageJ `spacing` field; the Y/X pixel sizes are
        the reciprocal of the TIFF Y/X resolution (stored as pixels-per-unit).
        Any value that is missing falls back to 1.0 so napari still renders.
        """
        z = y = x = 1.0
        if self.meta is not None:
            ij = self.meta.get("imagej") or {}
            spacing = ij.get("spacing")
            if spacing:
                z = float(spacing)
            yres = self.meta.get("yres")
            if yres:
                y = 1.0 / float(yres)
            xres = self.meta.get("xres")
            if xres:
                x = 1.0 / float(xres)
        return (z, y, x)

    # ------------------------------------------------------------------ paths
    def _curated_path(self):
        """Non-destructive full 3-channel safety net: `<base>_CURATED.tif`."""
        p = self.base_path
        return p.with_name(p.stem + "_CURATED" + p.suffix)

    def _mask_autosave_path(self):
        """Small mask-only autosave: `<base>_MASK_AUTOSAVE.tif`."""
        p = self.base_path
        return p.with_name(p.stem + "_MASK_AUTOSAVE" + p.suffix)

    def _mask_write_kwargs(self):
        """imagej kwargs for writing the mask-only (ZYX) autosave."""
        md = {"axes": "ZYX"}
        ij = self.meta["imagej"]
        for k in ("spacing", "unit"):
            if k in ij:
                md[k] = ij[k]
        kwargs = {"imagej": True, "metadata": md}
        if self.meta["xres"] and self.meta["yres"]:
            kwargs["resolution"] = (self.meta["xres"], self.meta["yres"])
        if self.meta["resunit"] is not None:
            kwargs["resolutionunit"] = self.meta["resunit"]
        return kwargs

    # --------------------------------------------------------------- regions
    @staticmethod
    def _split_axis(length, n_parts):
        """Split an axis into near-equal integer ranges [(start, end), ...]."""
        edges = np.linspace(0, int(length), int(n_parts) + 1, dtype=int)
        return [(int(edges[i]), int(edges[i + 1])) for i in range(int(n_parts))]

    def _choose_grid_shape(self, ny, nx):
        """2x2 (=4 regions) for smaller images, 3x3 (=9 regions) for large ones.

        "Large" means the smaller XY dimension is at least
        `REGION_GRID_THRESHOLD` pixels.
        """
        if min(int(ny), int(nx)) >= self.REGION_GRID_THRESHOLD:
            return 3, 3
        return 2, 2

    def _region_choices(self, widget=None):
        """Current region names for the dropdown.

        magicgui calls this (via `reset_choices`) to repopulate the combobox, so
        it always mirrors however many regions the loaded image was split into
        (4 or 9). Falls back to a 2x2 set before any image is loaded.
        """
        names = [name for name, *_ in self._region_bounds]
        return names or ["R1", "R2", "R3", "R4"]

    def _configure_regions(self, mask_zyx_shape):
        """Build region bounds and refresh region-widget choices."""
        _, ny, nx = [int(v) for v in mask_zyx_shape]
        self._region_rows, self._region_cols = self._choose_grid_shape(ny, nx)
        ys = self._split_axis(ny, self._region_rows)
        xs = self._split_axis(nx, self._region_cols)

        bounds = []
        idx = 1
        for r in range(self._region_rows):
            for c in range(self._region_cols):
                y0, y1 = ys[r]
                x0, x1 = xs[c]
                bounds.append((f"R{idx}", y0, y1, x0, x1))
                idx += 1
        self._region_bounds = bounds

        valid_names = [name for name, *_ in bounds]
        if self.active_region not in valid_names:
            self.active_region = valid_names[0]

        if self.region_widget is not None:
            # reset_choices re-invokes the callable choices so the dropdown
            # reflects the new region count (4 for small, 9 for large images).
            self.region_widget.region.reset_choices()
            self.region_widget.region.value = self.active_region

    def _get_region_bounds(self, region_name=None):
        """Return (name, y0, y1, x0, x1) for the requested region."""
        if not self._region_bounds:
            return None
        name = region_name if region_name is not None else self.active_region
        for item in self._region_bounds:
            if item[0] == name:
                return item
        return self._region_bounds[0]

    def _remove_region_outline(self):
        if self._region_outline_layer is None or self.viewer is None:
            self._region_outline_layer = None
            return
        try:
            self.viewer.layers.remove(self._region_outline_layer)
        except Exception:
            pass
        self._region_outline_layer = None

    def _draw_region_outline(self):
        """Draw a thick outline around the active region when scoped mode is enabled."""
        self._remove_region_outline()

        if self.viewer is None or self.mask_layer is None:
            return
        if self.operation_scope != "Selected region":
            return

        region = self._get_region_bounds()
        if region is None:
            return
        _, y0, y1, x0, x1 = region

        # Rectangle in YX (2D overlay, visible regardless of z-slice).
        rect = np.array([
            [y0, x0],
            [y0, max(x0 + 1, x1 - 1)],
            [max(y0 + 1, y1 - 1), max(x0 + 1, x1 - 1)],
            [max(y0 + 1, y1 - 1), x0],
        ], dtype=float)

        # The image/mask layers are scaled by the voxel size, so the outline
        # must share the same YX scale or it will be offset and too small.
        yx_scale = self._layer_scale()[1:]

        self._region_outline_layer = self.viewer.add_shapes(
            [rect],
            shape_type="rectangle",
            edge_color="yellow",
            edge_width=5,
            face_color="transparent",
            name="active region",
            ndim=2,
            scale=yx_scale,
        )
        self._region_outline_layer.editable = False

    def _set_operation_region(self, mode="Full image", region="R1"):
        """Update operation scope/region and refresh visual outline."""
        self.operation_scope = str(mode)
        self.active_region = str(region)

        if self.region_widget is not None:
            self.region_widget.region.enabled = (self.operation_scope == "Selected region")

        self._draw_region_outline()

        if self.operation_scope == "Full image":
            self._log("[REGION] Operations set to full XY extent")
        else:
            selected = self._get_region_bounds(self.active_region)
            if selected is not None:
                name, y0, y1, x0, x1 = selected
                self._log(f"[REGION] Operations set to {name}: y={y0}:{y1}, x={x0}:{x1}")

    def _get_operation_slices(self, data):
        """Return (ys, xs, description) for operation scope on data shaped (Z, Y, X)."""
        ny, nx = [int(v) for v in data.shape[-2:]]
        if self.operation_scope != "Selected region":
            return slice(0, ny), slice(0, nx), "full XY"

        selected = self._get_region_bounds(self.active_region)
        if selected is None:
            return slice(0, ny), slice(0, nx), "full XY"

        name, y0, y1, x0, x1 = selected
        return slice(y0, y1), slice(x0, x1), f"{name} (y={y0}:{y1}, x={x0}:{x1})"

    # -------------------------------------------------------------- undo/redo
    def _reset_history(self):
        """Clear the operation undo/redo history (called when a new image loads)."""
        self._undo_stack = []
        self._redo_stack = []
        self._update_history_buttons()

    def _update_history_buttons(self):
        """Enable/disable the undo/redo buttons to match the history stacks."""
        if self.undo_button is not None:
            self.undo_button.enabled = bool(self._undo_stack)
        if self.redo_button is not None:
            self.redo_button.enabled = bool(self._redo_stack)

    def _record_change(self, zmin, zmax, ys, xs, before):
        """Record one completed bulk edit so it can be undone/redone.

        Only the affected sub-volume (z range x ys x xs) is stored, as a
        before/after pair, so memory stays bounded even for large stacks. A new
        edit clears the redo stack (standard undo semantics).
        """
        data = np.asarray(self.mask_layer.data)
        after = data[zmin:zmax + 1, ys, xs].copy()
        self._undo_stack.append((zmin, zmax, ys, xs, before, after))
        if len(self._undo_stack) > self.MAX_HISTORY:
            self._undo_stack.pop(0)
        self._redo_stack.clear()
        self._update_history_buttons()

    def _apply_history_block(self, zmin, zmax, ys, xs, block):
        """Write a stored sub-volume back into the mask layer."""
        data = np.asarray(self.mask_layer.data)
        data[zmin:zmax + 1, ys, xs] = block
        self.mask_layer.data = data
        self.mask_layer.refresh()

    def _undo(self):
        """Revert the most recent bulk operation (replace / interpolate / fade)."""
        if self.mask_layer is None or not self._undo_stack:
            self._log("[UNDO] nothing to undo.")
            return
        entry = self._undo_stack.pop()
        zmin, zmax, ys, xs, before, after = entry
        self._apply_history_block(zmin, zmax, ys, xs, before)
        self._redo_stack.append(entry)
        self._dirty = True
        self._log(f"[UNDO] reverted operation on z={zmin}:{zmax}")
        self._update_history_buttons()
        self._save_mask_async()  # secure the reverted state

    def _redo(self):
        """Re-apply the most recently undone bulk operation."""
        if self.mask_layer is None or not self._redo_stack:
            self._log("[REDO] nothing to redo.")
            return
        entry = self._redo_stack.pop()
        zmin, zmax, ys, xs, before, after = entry
        self._apply_history_block(zmin, zmax, ys, xs, after)
        self._undo_stack.append(entry)
        self._dirty = True
        self._log(f"[REDO] re-applied operation on z={zmin}:{zmax}")
        self._update_history_buttons()
        self._save_mask_async()  # secure the re-applied state

    # ------------------------------------------------------------------ GUI
    def start(self):
        from qtpy.QtCore import QTimer

        self.viewer = napari.Viewer(title="Mask curation")

        self.load_widget = magicgui(
            self._load_image,
            image_path={"label": "Image", "mode": "r",
                        "filter": "TIFF (*.tif *.tiff)"},
            call_button="Load image",
        )
        if self.default_folder and self.default_folder.exists():
            self.load_widget.image_path.value = self.default_folder

        # Assign which channel is which role. brightfield + existing label are
        # required; cell / perfused fluorescence are optional ("None" = hidden).
        self.channel_widget = magicgui(
            self._apply_channel_assignment,
            call_button="Apply channel assignment",
            brightfield={"label": "Brightfield",
                         "choices": self._required_channel_choices},
            existing_label={"label": "Existing label",
                            "choices": self._required_channel_choices},
            cell_fluorescence={"label": "Fluorescence (cell)",
                               "choices": self._optional_channel_choices},
            perfused_fluorescence={"label": "Fluorescence (perfused)",
                                   "choices": self._optional_channel_choices},
        )
        self.channel_widget.brightfield.value = self.brightfield_channel
        self.channel_widget.existing_label.value = self.mask_channel
        self.channel_widget.cell_fluorescence.value = self.cell_fluo_channel
        self.channel_widget.perfused_fluorescence.value = self.perfused_channel

        # Size the channel dropdowns to the file's channel count as soon as it
        # is picked, so the allowed range is right before "Load image" is clicked.
        self.load_widget.image_path.changed.connect(self._on_image_path_changed)
        self._on_image_path_changed()

        self.region_widget = magicgui(
            self._set_operation_region,
            auto_call=True,
            mode={"label": "Operation scope", "choices": ["Full image", "Selected region"]},
            region={"label": "Region", "choices": self._region_choices},
        )

        # Replace a z-slice (or an inclusive range) of the MASK with another slice.
        self.replace_widget = magicgui(
            self._replace_slices,
            replace_z={"label": "Replace z (e.g. 16 or 1,3)"},
            source_z={"label": "with mask from z"},
            call_button="Replace slices",
        )

        # Interpolate the MASK between two z-slices (signed-distance morphing).
        self.interpolate_widget = magicgui(
            self._interpolate_slices,
            z_min={"label": "z min"},
            z_max={"label": "z max"},
            call_button="Interpolate",
        )

        # Fade out (shrink to empty) the MASK from one z-slice to another.
        self.fade_widget = magicgui(
            self._fade_out_slices,
            z_start={"label": "fade out from z"},
            z_end={"label": "to z"},
            call_button="Fade out",
        )

        # Undo / redo for the bulk operations above (replace / interpolate /
        # fade out). This is separate from napari's built-in paint/erase undo.
        self.undo_button = PushButton(text="Undo last operation")
        self.redo_button = PushButton(text="Redo")
        self.undo_button.clicked.connect(self._undo)
        self.redo_button.clicked.connect(self._redo)
        self.history_widget = Container(
            widgets=[self.undo_button, self.redo_button],
            layout="horizontal", labels=False,
        )

        # Manual FINAL save = COMMIT the 3-channel image over the original.
        self.save_widget = magicgui(self._save_full,
                                    call_button="Save final (overwrite original)")

        self.log_widget = TextEdit(value="", label="Log")
        try:
            self.log_widget.native.setReadOnly(True)
        except Exception:
            pass
        self.log_widget.min_height = 120
        self.log_widget.max_height = 300

        self.viewer.window.add_dock_widget(self.load_widget, area="right",
                                           name="Select image")
        self.viewer.window.add_dock_widget(self.channel_widget, area="right",
                                           name="Channel assignment")
        self.viewer.window.add_dock_widget(self.region_widget, area="right",
                                           name="Operation region")
        self.viewer.window.add_dock_widget(self.replace_widget, area="right",
                                           name="Replace slices")
        self.viewer.window.add_dock_widget(self.interpolate_widget, area="right",
                                           name="Interpolate between")
        self.viewer.window.add_dock_widget(self.fade_widget, area="right",
                                           name="Fade out")
        self.viewer.window.add_dock_widget(self.history_widget, area="right",
                                           name="Undo / redo operations")
        self.viewer.window.add_dock_widget(self.save_widget, area="right",
                                           name="Save")
        self.viewer.window.add_dock_widget(self.log_widget, area="right",
                                           name="Log")

        # Autosave (mask only, background thread) on a timer.
        self._autosave_timer = QTimer()
        self._autosave_timer.setInterval(self.AUTOSAVE_SECONDS * 1000)
        self._autosave_timer.timeout.connect(self._autosave)
        self._autosave_timer.start()

        # Finalise (full 3-channel save) when the window is closed.
        qt_window = self.viewer.window._qt_window
        self._orig_close = qt_window.closeEvent
        qt_window.closeEvent = self._on_close

        # Ensure region UI starts in a valid state before loading data.
        self._set_operation_region(self.operation_scope, self.active_region)

        # Undo/redo buttons start disabled until an operation is performed.
        self._update_history_buttons()

    # ------------------------------------------------------------------ load
    def _load_image(self, image_path=Path()):
        image_path = Path(image_path)
        if not image_path.is_file():
            self._log("[WARN] Please select a tif file.")
            return

        # Identity/base name (strip a _CURATED suffix if the user picked one).
        stem = image_path.stem
        if stem.endswith("_CURATED"):
            self.base_path = image_path.with_name(
                stem[: -len("_CURATED")] + image_path.suffix)
        else:
            self.base_path = image_path

        # Prefer an existing full _CURATED as the source of pixel data.
        curated = self._curated_path()
        source = curated if curated.is_file() else self.base_path
        if source == curated:
            self._log(f"[RESUME] Loading full progress from {curated.name}")

        try:
            arr, meta = read_image_and_meta(source)
        except Exception as e:
            self._log(f"[ERROR] Could not read {source.name}: "
                      f"{type(e).__name__}: {e}")
            return

        self.path = source
        self.arr, self.meta = arr, meta

        # Refresh the channel-assignment dropdowns to match this stack, then read
        # back the (possibly adjusted) role selections before anything uses them.
        self._num_channels = int(self.arr.shape[1]) if self.arr.ndim >= 2 else 1
        if self.channel_widget is not None:
            self.channel_widget.reset_choices()
        self._read_channel_assignment()

        # If a mask-only autosave exists that is newer than the ORIGINAL file,
        # overlay it onto the mask channel. The comparison is against the original
        # (base_path) rather than the loaded source, so a _CURATED sidecar can't
        # hide a newer autosave.
        ma = self._mask_autosave_path()
        original_mtime = (self.base_path.stat().st_mtime
                          if self.base_path.is_file() else source.stat().st_mtime)
        if ma.is_file() and ma.stat().st_mtime > original_mtime:
            try:
                mask_arr = tifffile.imread(ma)
                expected = self.arr[:, self.mask_channel].shape
                if mask_arr.shape == expected:
                    # Direct assignment casts in place -> no full-size temp copy.
                    self.arr[:, self.mask_channel] = mask_arr
                    self._log(f"[RESUME] Applied newer mask autosave {ma.name}")
                else:
                    self._log(f"[WARN] Mask autosave shape {mask_arr.shape} != "
                              f"expected {expected}; ignored.")
                del mask_arr
            except Exception as e:
                self._log(f"[WARN] Could not read mask autosave: {e}")

        # Build the display layers from the current channel-role assignment.
        if not self._build_layers():
            return

        self.viewer.title = self.base_path.name
        self._dirty = False
        self._reset_history()  # fresh image -> clear operation undo/redo history
        self._log(f"[OK] Loaded {source.name}\n"
                  f"  mask autosave -> {self._mask_autosave_path().name}\n"
                  f"  save final    -> overwrites {self.base_path.name}")

    # -------------------------------------------------------------- channels
    @staticmethod
    def _peek_num_channels(path):
        """Read the channel count from a tif's header WITHOUT loading the pixels."""
        try:
            with tifffile.TiffFile(path) as tif:
                series = max(tif.series, key=lambda s: s.size)
                axes, shape = series.axes, series.shape
            if "C" in axes:
                return int(shape[axes.index("C")])
            if len(shape) >= 4:      # fall back to the (Z, C, Y, X) convention
                return int(shape[1])
        except Exception:
            return None
        return None

    def _on_image_path_changed(self, *_):
        """Resize the channel dropdowns to match the currently selected file."""
        if self.channel_widget is None:
            return
        path = Path(self.load_widget.image_path.value)
        if not path.is_file():
            return
        n = self._peek_num_channels(path)
        if n and n != self._num_channels:
            self._num_channels = n
            self.channel_widget.reset_choices()
            self._log(f"[CHANNELS] {path.name} has {n} channels; dropdowns updated.")

    def _required_channel_choices(self, widget=None):
        """Channel indices available for a required role (brightfield / label)."""
        n = self._num_channels or 3
        return [(str(i), i) for i in range(n)]

    def _optional_channel_choices(self, widget=None):
        """Channel indices for an optional role, plus a "None" (not shown) entry."""
        n = self._num_channels or 3
        return [("None", None)] + [(str(i), i) for i in range(n)]

    def _read_channel_assignment(self):
        """Pull the current role -> channel selections from the GUI dropdowns."""
        if self.channel_widget is None:
            return
        self.brightfield_channel = self.channel_widget.brightfield.value
        self.mask_channel = self.channel_widget.existing_label.value
        self.cell_fluo_channel = self.channel_widget.cell_fluorescence.value
        self.perfused_channel = self.channel_widget.perfused_fluorescence.value

    def _apply_channel_assignment(self, brightfield=0, existing_label=2,
                                  cell_fluorescence=None, perfused_fluorescence=None):
        """Assign which channel plays each role and (re)build the display layers.

        `brightfield` and `existing_label` are required; the cell / perfused
        fluorescence channels are optional (None -> that layer is not shown).
        """
        # Preserve any in-progress mask edits before rebuilding the layers.
        if self.arr is not None and self.mask_layer is not None and self._dirty:
            self._sync_mask_into_arr()

        self.brightfield_channel = brightfield
        self.mask_channel = existing_label
        self.cell_fluo_channel = cell_fluorescence
        self.perfused_channel = perfused_fluorescence

        if self.arr is None:
            self._log("[CHANNELS] Assignment saved; it applies when you load an image.")
            return
        self._build_layers()

    def _build_layers(self):
        """(Re)build the napari layers from `self.arr` using the current channel
        roles. brightfield + existing label are required; the cell / perfused
        fluorescence layers are only added when their channel is set.

        Returns True on success, False if a required channel is invalid.
        """
        if self.arr is None or self.viewer is None:
            return False

        n = int(self.arr.shape[1])

        def _valid(ch):
            return ch is not None and 0 <= int(ch) < n

        if not _valid(self.brightfield_channel):
            self._log(f"[ERROR] Brightfield channel {self.brightfield_channel} is "
                      f"out of range (image has {n} channels).")
            return False
        if not _valid(self.mask_channel):
            self._log(f"[ERROR] Existing-label channel {self.mask_channel} is "
                      f"out of range (image has {n} channels).")
            return False

        scale = self._layer_scale()
        self._log(f"[OK] Using voxel scale (z, y, x) = {scale}")

        # Replace any layers from a previously loaded image / assignment.
        self._region_outline_layer = None
        self.viewer.layers.clear()

        brightfield = self.arr[:, self.brightfield_channel, :, :]
        self.viewer.add_image(brightfield, name="brightfield",
                              colormap="gray", blending="additive", scale=scale,
                              contrast_limits=self._contrast_limits(brightfield))

        if _valid(self.cell_fluo_channel):
            cell = self.arr[:, self.cell_fluo_channel, :, :]
            self.viewer.add_image(cell, name="fluorescence (cell)",
                                  colormap="green", blending="additive", scale=scale,
                                  contrast_limits=self._contrast_limits(cell))

        if _valid(self.perfused_channel):
            perfused = self.arr[:, self.perfused_channel, :, :]
            self.viewer.add_image(perfused, name="fluorescence (perfused)",
                                  colormap="magenta", blending="additive", scale=scale,
                                  contrast_limits=self._contrast_limits(perfused))

        mask = self.arr[:, self.mask_channel, :, :]
        # Hold the mask as the smallest integer dtype that fits its labels
        # (uint8/uint16) rather than int32, to keep the RAM footprint low on
        # these large XY stacks.
        labels_dtype = self._labels_dtype(mask.max())
        self.mask_layer = self.viewer.add_labels(mask.astype(labels_dtype),
                                                 name="mask", scale=scale)
        # Mark unsaved whenever the mask is painted/edited.
        self.mask_layer.events.paint.connect(self._mark_dirty)
        self.mask_layer.events.data.connect(self._mark_dirty)

        # Region setup depends on loaded XY size.
        self._configure_regions(mask.shape)
        self._set_operation_region(self.operation_scope, self.active_region)

        self.viewer.layers.selection.active = self.mask_layer
        self._log(f"[OK] Channels -> brightfield={self.brightfield_channel}, "
                  f"existing label={self.mask_channel}, "
                  f"cell={self.cell_fluo_channel}, perfused={self.perfused_channel}")
        return True

    # ------------------------------------------------------------------ edits
    @staticmethod
    def _parse_z_range(text, nz):
        """'16' -> [16];  '1,3' or '1-3' -> [1,2,3] (inclusive). None if invalid."""
        t = str(text).strip().strip("{}[]()")
        for sep in ("-", " ", ";"):
            t = t.replace(sep, ",")
        parts = [p for p in t.split(",") if p != ""]
        if not parts:
            return None
        try:
            nums = [int(p) for p in parts]
        except ValueError:
            return None
        lo, hi = (nums[0], nums[0]) if len(nums) == 1 else (nums[0], nums[1])
        if lo > hi:
            lo, hi = hi, lo
        if lo < 0 or hi >= nz:
            return None
        return list(range(lo, hi + 1))

    @staticmethod
    def _parse_single_z(text, nz):
        t = str(text).strip().strip("{}[]()")
        try:
            z = int(t)
        except ValueError:
            return None
        return z if 0 <= z < nz else None

    def _replace_slices(self, replace_z: str = "", source_z: str = ""):
        """Copy the MASK from `source_z` into every z in `replace_z`.

        Operates ONLY on the mask layer (channel `mask_channel`); the brightfield
        and fluorescence layers are never touched.
        """
        if self.mask_layer is None:
            self._log("[WARN] Load an image before replacing slices.")
            return

        data = np.asarray(self.mask_layer.data)
        nz = data.shape[0]

        targets = self._parse_z_range(replace_z, nz)
        if targets is None:
            self._log(f"[ERROR] 'Replace z' = {replace_z!r} is invalid "
                      f"(use e.g. 16 or 1,3; valid range 0..{nz - 1}).")
            return
        src = self._parse_single_z(source_z, nz)
        if src is None:
            self._log(f"[ERROR] 'with mask from z' = {source_z!r} is invalid "
                      f"(use a single slice 0..{nz - 1}).")
            return

        ys, xs, scope_desc = self._get_operation_slices(data)
        zmin, zmax = min(targets), max(targets)
        before = data[zmin:zmax + 1, ys, xs].copy()
        src_slice = data[src, ys, xs].copy()
        for z in targets:
            data[z, ys, xs] = src_slice
        self.mask_layer.data = data
        self.mask_layer.refresh()
        self._record_change(zmin, zmax, ys, xs, before)

        self._dirty = True
        if len(targets) == 1:
            self._log(f"[OK] Successfully replaced mask slice {targets[0]} "
                      f"with slice {src} in {scope_desc}")
        else:
            self._log(f"[OK] Successfully replaced mask slices "
                      f"{targets[0]}-{targets[-1]} with slice {src} in {scope_desc}")
        self._save_mask_async()  # secure mask progress straight away

    @staticmethod
    def _signed_distance(mask):
        """Signed distance transform: positive inside the mask, negative outside."""
        mask = mask.astype(bool)
        inside = distance_transform_edt(mask)
        outside = distance_transform_edt(~mask)
        return inside - outside

    @classmethod
    def _interpolate_binary_masks(cls, mask_start, mask_end, n_slices):
        """SDF-morph between two binary masks; returns `n_slices` binary slices
        with the endpoints reproduced exactly."""
        sdf_start = cls._signed_distance(mask_start)
        sdf_end = cls._signed_distance(mask_end)
        out = np.zeros((n_slices, *mask_start.shape), dtype=np.uint8)
        for i in range(n_slices):
            t = i / (n_slices - 1)
            sdf = (1.0 - t) * sdf_start + t * sdf_end
            out[i] = (sdf >= 0).astype(np.uint8)
        out[0] = mask_start.astype(bool).astype(np.uint8)
        out[-1] = mask_end.astype(bool).astype(np.uint8)
        return out

    def _interpolate_slices(self, z_min: str = "", z_max: str = ""):
        """Interpolate the MASK between slices `z_min` and `z_max` (inclusive),
        morphing the shape from one to the other. The two endpoint slices are
        kept exactly; every slice in between is filled in.

        Operates ONLY on the mask layer (channel `mask_channel`).
        """
        if self.mask_layer is None:
            self._log("[WARN] Load an image before interpolating.")
            return

        data = np.asarray(self.mask_layer.data)
        nz = data.shape[0]

        lo = self._parse_single_z(z_min, nz)
        if lo is None:
            self._log(f"[ERROR] 'z min' = {z_min!r} is invalid "
                      f"(use a single slice 0..{nz - 1}).")
            return
        hi = self._parse_single_z(z_max, nz)
        if hi is None:
            self._log(f"[ERROR] 'z max' = {z_max!r} is invalid "
                      f"(use a single slice 0..{nz - 1}).")
            return
        if lo == hi:
            self._log("[ERROR] z min and z max must be different slices.")
            return
        if lo > hi:
            lo, hi = hi, lo

        ys, xs, scope_desc = self._get_operation_slices(data)
        before = data[lo:hi + 1, ys, xs].copy()
        mask_start = data[lo, ys, xs]
        mask_end = data[hi, ys, xs]
        interp = self._interpolate_binary_masks(mask_start, mask_end, hi - lo + 1)

        # Preserve the mask's label value in the operated scope.
        label_val = int(max(int(mask_start.max()), int(mask_end.max()))) or 1
        data[lo:hi + 1, ys, xs] = (interp * label_val).astype(data.dtype)
        self.mask_layer.data = data
        self.mask_layer.refresh()
        self._record_change(lo, hi, ys, xs, before)

        self._dirty = True
        self._log(f"[OK] Successfully interpolated mask slices {lo}-{hi} "
                  f"(endpoints {lo} and {hi} kept) in {scope_desc}")
        self._save_mask_async()  # secure mask progress straight away

    @staticmethod
    def _shrink_mask_to_empty(mask, number_of_slices):
        """Progressively shrink a binary mask to empty over `number_of_slices`
        slices using a distance-transform threshold (fast; NOT iterative
        erosion). Slice 0 is the original mask, the last slice is all-empty.
        """
        if number_of_slices < 2:
            raise ValueError("number_of_slices must be at least 2.")
        mask = mask.astype(bool)
        output = np.zeros((number_of_slices, *mask.shape), dtype=np.uint8)
        if not mask.any():
            return output
        # Distance of each foreground pixel from the nearest background pixel.
        distance = distance_transform_edt(mask)
        maximum_distance = distance.max()
        for index in range(number_of_slices):
            t = index / (number_of_slices - 1)
            if index == 0:
                output[index] = mask
            elif index == number_of_slices - 1:
                output[index] = 0
            else:
                threshold = t * maximum_distance
                output[index] = (distance > threshold).astype(np.uint8)
        return output

    def _fade_out_slices(self, z_start: str = "", z_end: str = ""):
        """Fade the MASK out from slice `z_start` (kept as-is) to slice `z_end`
        (set fully empty), progressively shrinking the shape in between via a
        fast distance-transform threshold.

        Operates ONLY on the mask layer (channel `mask_channel`). Direction is
        honoured: `z_end` can be above or below `z_start`.
        """
        if self.mask_layer is None:
            self._log("[WARN] Load an image before fading out.")
            return

        data = np.asarray(self.mask_layer.data)
        nz = data.shape[0]

        start = self._parse_single_z(z_start, nz)
        if start is None:
            self._log(f"[ERROR] 'fade out from z' = {z_start!r} is invalid "
                      f"(use a single slice 0..{nz - 1}).")
            return
        end = self._parse_single_z(z_end, nz)
        if end is None:
            self._log(f"[ERROR] 'to z' = {z_end!r} is invalid "
                      f"(use a single slice 0..{nz - 1}).")
            return
        if start == end:
            self._log("[ERROR] fade-out start and end must be different slices.")
            return

        ys, xs, scope_desc = self._get_operation_slices(data)
        zmin, zmax = min(start, end), max(start, end)
        before = data[zmin:zmax + 1, ys, xs].copy()
        mask = data[start, ys, xs]
        label_val = int(mask.max()) or 1
        n = abs(end - start) + 1
        faded = self._shrink_mask_to_empty(mask, n)  # faded[0]=mask, faded[-1]=empty

        # Walk from start toward end so faded[0] lands on z_start, faded[-1] on z_end.
        step = 1 if end > start else -1
        for k, z in enumerate(range(start, end + step, step)):
            data[z, ys, xs] = (faded[k] * label_val).astype(data.dtype)

        self.mask_layer.data = data
        self.mask_layer.refresh()
        self._record_change(zmin, zmax, ys, xs, before)

        self._dirty = True
        self._log(f"[OK] Successfully faded out mask from z={start} to z={end} "
                  f"(z={end} set empty) in {scope_desc}")
        self._save_mask_async()  # secure mask progress straight away

    # ------------------------------------------------------------------ save
    def _mark_dirty(self, event=None):
        self._dirty = True

    def _sync_mask_into_arr(self):
        # Direct assignment casts element-wise into the existing arr buffer, so
        # it avoids allocating a second full-size copy (the previous .astype()
        # made a whole extra array, which could exhaust RAM on big stacks).
        self.arr[:, self.mask_channel, :, :] = self.mask_layer.data

    def _on_save_finished(self):
        """Runs on the GUI thread when any background save worker ends."""
        self._save_running = False
        self._save_worker = None

    def _save_mask_async(self):
        """Autosave ONLY the mask channel (ZYX) on a background thread."""
        if self.mask_layer is None or self.base_path is None:
            return
        if self._save_running:
            self._log("[autosave] previous save still running; will retry next tick.")
            return

        # Snapshot on the GUI thread so the worker has a stable, private copy.
        # Keep the layer's own compact dtype (uint8/uint16) instead of upcasting
        # to arr.dtype -- a single, as-small-as-possible copy.
        mask_snapshot = np.array(self.mask_layer.data)
        out = self._mask_autosave_path()
        kwargs = self._mask_write_kwargs()
        self._dirty = False  # captured in snapshot; worker re-flags on failure

        @thread_worker
        def _work():
            safe_imwrite(out, mask_snapshot, **kwargs)
            return out.name

        def _ok(name):
            self._log(f"[autosave] save completed -> {name}")

        def _err(exc):
            self._dirty = True  # force a retry on the next tick
            self._log(f"[autosave][ERROR] {type(exc).__name__}: {exc}")

        worker = _work()
        worker.returned.connect(_ok)
        worker.errored.connect(_err)
        worker.finished.connect(self._on_save_finished)
        self._save_worker = worker
        self._save_running = True
        worker.start()
        self._log(f"[autosave] writing mask in background -> {out.name} ...")

    def _autosave(self):
        if self._dirty and self.mask_layer is not None and self.base_path is not None:
            self._save_mask_async()

    def _save_full(self):
        """FINAL save / COMMIT: fold the mask into the 3-channel array, write it
        OVER the original file, then delete the now-obsolete _CURATED and
        _MASK_AUTOSAVE sidecars. Runs on a background thread.
        """
        if self.mask_layer is None or self.arr is None:
            self._log("[WARN] Load an image before saving.")
            return
        if self._save_running:
            self._log("[SAVE] a save is already in progress; try again shortly.")
            return

        self._sync_mask_into_arr()  # fold current mask into arr on the GUI thread
        self._dirty = False          # snapshot taken; a later edit re-flags via paint
        out = self.base_path                 # COMMIT over the original file
        arr, meta = self.arr, self.meta
        curated = self._curated_path()
        ma = self._mask_autosave_path()

        @thread_worker
        def _work():
            save_image(out, arr, meta)
            # original now holds the curated result -> remove obsolete sidecars
            for side in (curated, ma):
                try:
                    if side != out and side.is_file():
                        side.unlink()
                except Exception:
                    pass
            return out.name

        def _ok(name):
            self.path = self.base_path
            self._log(f"[SAVE] save completed -> {name} "
                      f"(original overwritten; sidecars removed)")

        def _err(exc):
            self._dirty = True  # commit failed -> keep marked so it can be re-saved
            self._log(f"[SAVE][ERROR] {type(exc).__name__}: {exc}; "
                      f"original + sidecars left intact.")

        worker = _work()
        worker.returned.connect(_ok)
        worker.errored.connect(_err)
        worker.finished.connect(self._on_save_finished)
        self._save_worker = worker
        self._save_running = True
        worker.start()
        self._log(f"[SAVE] writing full 3-channel OVER original -> {out.name} "
                  f"(this can take a while)...")

    def _on_close(self, event):
        """On close: if there are unsaved mask edits, write the full 3-channel
        image SYNCHRONOUSLY to the non-destructive _CURATED safety net (so the
        process cannot exit mid-write and the original is untouched). If nothing
        changed since the last save, skip the write entirely."""
        if self._autosave_timer is not None:
            self._autosave_timer.stop()

        if not self._dirty:
            self._log("[CLOSE] no unsaved changes; nothing to save.")
            self._orig_close(event)
            return

        if self.mask_layer is not None and self.arr is not None:
            try:
                self._sync_mask_into_arr()
                out = self._curated_path()
                self._log(f"[CLOSE] saving progress -> {out.name} ...")
                save_image(out, self.arr, self.meta)
                self._dirty = False
                self._log(f"[CLOSE] save completed -> {out.name}")
            except Exception as e:
                self._log(f"[CLOSE][ERROR] {type(e).__name__}: {e}; "
                          f"mask autosave kept.")
        self._orig_close(event)


In [22]:
# Default folder the file explorer opens in (you can change to wherever your images live, or just navigate to the folder each time from the GUI).
masks_folder = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\3_channel_images_to_curate")

In [23]:
curator = MaskCurator(default_folder=masks_folder)
curator.start()

In [24]:
# input_path = Path(r"Z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\3_channel_images_to_curate\CART_day7_FL32_FL32_ARi2_Merged.tif")
# image = tifffile.imread(input_path)
# print(image.shape)

In [25]:

# def signed_distance(mask: np.ndarray) -> np.ndarray:
#     """
#     Positive inside the mask, negative outside.
#     """
#     mask = mask.astype(bool)
#     distance_inside = distance_transform_edt(mask)
#     distance_outside = distance_transform_edt(~mask)
#     return distance_inside - distance_outside



# z_end = 20
# z_start = 2
# mask_start = image[z_start,2,:,:]
# mask_end = image[z_end,2,:,:]
# sdf_start = signed_distance(mask_start)
# sdf_end = signed_distance(mask_end)

# number_of_slices = z_end - z_start + 1
# interpolated = np.zeros((number_of_slices, *mask_start.shape), dtype=np.uint8)

# for index in range(number_of_slices):
#     t = index / (number_of_slices - 1)
#     interpolated_sdf = ((1.0 - t) * sdf_start + t * sdf_end)
#     interpolated[index] = (interpolated_sdf >= 0).astype(np.uint8)

# # Ensure the supplied endpoint masks are reproduced exactly
# interpolated[0] = mask_start.astype(np.uint8)
# interpolated[-1] = mask_end.astype(np.uint8)

# fig, ax = plt.subplots(ncols = 3)
# ax[0].imshow(mask_start)
# ax[1].imshow(image[11,2,:,:])
# ax[2].imshow(mask_end)
# plt.show()

# fig, ax = plt.subplots(ncols = 3)
# ax[0].imshow(interpolated[0])
# ax[1].imshow(interpolated[-1])
# ax[2].imshow(interpolated[9])


In [26]:
# viewer = napari.Viewer()
# viewer.add_labels(image[:,2,:,:])
# viewer.add_labels(interpolated)

In [27]:

# def interpolate_binary_masks(mask_start: np.ndarray, mask_end: np.ndarray, z_start: int, z_end: int,) -> np.ndarray:
#     """
#     Interpolate binary masks between two known z-slices.

#     Returns an array with shape:
#         (z_end - z_start + 1, height, width)

#     The first and last slices are the supplied masks.
#     """
#     if mask_start.shape != mask_end.shape:
#         raise ValueError("The two masks must have the same shape.")

#     if z_end <= z_start:
#         raise ValueError("z_end must be greater than z_start.")

#     sdf_start = signed_distance(mask_start)
#     sdf_end = signed_distance(mask_end)

#     number_of_slices = z_end - z_start + 1
#     interpolated = np.zeros((number_of_slices, *mask_start.shape), dtype=np.uint8)

#     for index in range(number_of_slices):
#         t = index / (number_of_slices - 1)
#         interpolated_sdf = ((1.0 - t) * sdf_start + t * sdf_end)
#         interpolated[index] = (interpolated_sdf >= 0).astype(np.uint8)

#     # Ensure the supplied endpoint masks are reproduced exactly
#     interpolated[0] = mask_start.astype(np.uint8)
#     interpolated[-1] = mask_end.astype(np.uint8)

#     return interpolated

In [28]:
# for z in range(18, image.shape[0]):
#     image[z,2,:,:] = image[17,2,:,:]

In [29]:
# import numpy as np
# from scipy.ndimage import distance_transform_edt


# def shrink_mask_to_empty(
#     mask: np.ndarray,
#     number_of_slices: int,
# ) -> np.ndarray:
#     """
#     Generate slices that progressively shrink a binary mask to an empty mask.

#     Parameters
#     ----------
#     mask
#         2D binary mask.
#     number_of_slices
#         Total number of slices, including the original and empty endpoints.

#     Returns
#     -------
#     np.ndarray
#         Shape: (number_of_slices, height, width)
#     """
#     if number_of_slices < 2:
#         raise ValueError("number_of_slices must be at least 2.")

#     mask = mask.astype(bool)

#     output = np.zeros(
#         (number_of_slices, *mask.shape),
#         dtype=np.uint8,
#     )

#     if not mask.any():
#         return output

#     # Distance of each foreground pixel from the nearest background pixel
#     distance = distance_transform_edt(mask)
#     maximum_distance = distance.max()

#     for index in range(number_of_slices):
#         t = index / (number_of_slices - 1)

#         if index == 0:
#             output[index] = mask
#         elif index == number_of_slices - 1:
#             output[index] = 0
#         else:
#             threshold = t * maximum_distance
#             output[index] = (distance > threshold).astype(np.uint8)

#     return output

# In Case of Memory Error

In [30]:
# import tifffile, numpy as np

# layer = curator.mask_layer.data          # int32 array already in RAM (no copy made)
# mx = int(np.asarray(layer).max())
# dtype = np.uint8 if mx <= 255 else (np.uint16 if mx <= 65535 else np.int32)

# rescue = curator.base_path.with_name(curator.base_path.stem + "_MASK_RESCUE.tif")
# with tifffile.TiffWriter(rescue) as tif:
#     for z in range(layer.shape[0]):
#         plane = np.ascontiguousarray(layer[z]).astype(dtype)  # ~31 MiB, one slice only
#         tif.write(plane, contiguous=True)
#         del plane
# print("rescued mask ->", rescue)

In [31]:
# # Merge a 3-channel image with a mask-only autosave, writing the result to a NEW file.
# image_path = Path(r"z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\3_channel_images_to_curate\Bel_mCh_BF_VASCUMAP_FL33_ARi1_Merged.tif")
# mask_path = Path(r"z:\Bel\IMAGES_FOR_VASCUMAP_RETRAINING\fluorescent_cells_tifs\3_channel_images_to_curate\Bel_mCh_BF_VASCUMAP_FL33_ARi1_Merged_MASK_AUTOSAVE.tif")
# mask_channel = 2  # channel index the curated mask lives in (matches MaskCurator default)

# arr, meta = read_image_and_meta(image_path)   # (Z, C, Y, X)
# mask_arr = tifffile.imread(mask_path)          # mask-only (Z, Y, X)

# expected = arr[:, mask_channel].shape
# if mask_arr.shape != expected:
#     raise ValueError(f"Mask shape {mask_arr.shape} != expected {expected} for channel {mask_channel}.")

# # Overlay the autosave mask onto the mask channel (cast in place, no full-size copy).
# arr[:, mask_channel] = mask_arr
# del mask_arr

# # New output path (never overwrites the originals).
# out_path = image_path.with_name(image_path.stem + "_MERGED_WITH_AUTOSAVE" + image_path.suffix)
# save_image(out_path, arr, meta)
# print("merged ->", out_path)
